# RAG 평가셋

`data/evaluation/teamcandidates/candidates유건.jsonl`을 평가 질문으로 사용하고, `data/processed/chunks.jsonl`을 검색 대상으로 삼습니다. 답변 충실도, 답변 관련성, 답변 정확성, 컨텍스트 정밀도, 컨텍스트 재현율, 컨텍스트 관련성을 Ragas로 평가합니다.

현재는 LangChain 임시 답변 생성기를 사용합니다. Agent가 완성되면 아래 `answer_generator(question, retrieved_chunks)` 계약을 지키는 함수로 교체하면 됩니다. `expected_facts`는 참조 답변, `expected_source_chunk_ids`는 참조 컨텍스트이며 검색 결과와 섞지 않습니다.

## 1단계: 평가 대상과 기준값 읽기

이 셀에서는 평가 질문 파일을 읽습니다. 각 후보에는 질문(`question`), 기대 답변에 포함되어야 할 사실(`expected_facts`), 정답 근거 문서의 ID(`expected_source_chunk_ids`)가 들어 있습니다.

평가에서 이 둘은 서로 다른 용도로 사용됩니다.

- `expected_facts`: 생성된 답변이 맞는지 비교하는 참조 답변
- `expected_source_chunk_ids`: 검색 결과가 정답 근거를 얼마나 잘 찾았는지 비교하는 참조 컨텍스트

즉, 검색 결과를 정답으로 간주하지 않고, 후보셋의 기준값과 별도로 비교합니다.

In [1]:
from __future__ import annotations
import json, os, re
from collections import defaultdict
from pathlib import Path
import pandas as pd
from dotenv import find_dotenv, load_dotenv

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data").exists(): ROOT = ROOT.parent
load_dotenv(find_dotenv(usecwd=True)); load_dotenv(ROOT.parent / ".env")
CANDIDATE_PATH = ROOT / "data/evaluation/teamcandidates/candidates유건.jsonl"
CHUNK_PATH = ROOT / "data/processed/chunks.jsonl"
TOP_K = int(os.getenv("RAG_EVAL_TOP_K", "5")); MODEL_NAME = os.getenv("OPENAI_EVAL_MODEL", "gpt-4o-mini")
def read_jsonl(path):
    with path.open(encoding="utf-8") as f: return [json.loads(x) for x in f if x.strip()]
# 평가 질문과 기대 정답을 읽습니다.
samples = read_jsonl(CANDIDATE_PATH)
print(f"questions={len(samples)}")

python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 5


questions=15


## 2단계: 검색기 구성

`data/processed/chunks.jsonl`의 각 chunk를 읽어 간단한 역색인을 만듭니다.

질문에 포함된 단어가 어떤 chunk에 등장하는지 찾고, 질문과 겹치는 단어 수가 많은 순서로 `TOP_K`개를 반환합니다. 이것은 현재 Agent가 없을 때 사용할 수 있는 임시 검색기입니다.

실제 팀 검색기나 벡터 검색기가 완성되면 `retrieve()` 함수만 교체하면 됩니다. 이후 평가 코드는 검색기의 내부 구현을 알 필요가 없습니다.

검색 결과에는 `chunk_id`와 본문을 함께 넣어 LLM이 어떤 근거를 사용했는지 추적할 수 있게 했습니다.

In [2]:
# 독립적인 간단 검색기. 기존 팀 RAG 코드의 객체/설정은 변경하지 않습니다.
TOKEN_RE = re.compile(r"[^\W_]{2,}", re.UNICODE)
# 원천 chunk를 모두 읽고 검색용 인덱스를 준비합니다.
chunks = read_jsonl(CHUNK_PATH); chunk_by_id = {c["chunk_id"]: c for c in chunks if c.get("chunk_id")}
inverted, chunk_tokens = defaultdict(list), []
for i, c in enumerate(chunks):
    toks = set(TOKEN_RE.findall((c.get("embedding_text") or c.get("clean_text") or "").lower())); chunk_tokens.append(toks)
    for t in toks: inverted[t].append(i)
def retrieve(question, top_k=TOP_K):
    q = set(TOKEN_RE.findall(question.lower())); candidates = {i for t in q for i in inverted.get(t, ())}
    scored = [(len(q & chunk_tokens[i]) / max(len(q), 1), len(q & chunk_tokens[i]), i) for i in candidates if q & chunk_tokens[i]]
    scored.sort(reverse=True); return [chunks[i] for _, _, i in scored[:top_k]]
def context_text(c): return f"[chunk_id={c.get('chunk_id')}]\n{c.get('clean_text') or c.get('raw_text') or ''}"
print(f"chunks={len(chunks)}")

chunks=30661


## 3단계: 임시 LangChain 답변 생성기

현재 Agent가 없으므로 `ChatOpenAI`를 LangChain 체인으로 감싼 임시 답변 생성기를 사용합니다.

입력은 다음 두 가지입니다.

1. 평가 질문
2. 검색기가 반환한 실제 컨텍스트

LLM은 컨텍스트만 사용해 답변하도록 지시받습니다. 근거가 없으면 `INSUFFICIENT_EVIDENCE`를 반환하도록 하여, 모르는 내용을 추측하는 답변을 줄입니다.

나중에 Agent가 완성되면 `answer_generator`가 같은 형태로 `질문 + 검색 결과 → 답변 문자열`을 반환하도록 교체하면 됩니다. 그러면 아래 평가 로직은 수정하지 않아도 됩니다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
class LangChainRAGAnswerGenerator:
    def __init__(self, model=MODEL_NAME):
        self.llm = ChatOpenAI(model=model, temperature=0)
        self.chain = ChatPromptTemplate.from_messages([
            ("system", "근거 기반 한국어 RAG 답변기다. 컨텍스트만 사용하고 근거가 없으면 INSUFFICIENT_EVIDENCE라고 답하라. 추측하지 말라."),
            ("human", "질문: {question}\n\n컨텍스트:\n{context}")]) | self.llm
    def __call__(self, question, retrieved_chunks):
        return self.chain.invoke({"question": question, "context": "\n\n".join(context_text(c) for c in retrieved_chunks)}).content
answer_generator = LangChainRAGAnswerGenerator()
# Agent 교체 예: answer_generator = lambda q, cs: completed_agent.invoke({"question": q, "contexts": cs})["answer"]

## 4단계: 평가용 데이터셋 만들기

각 질문에 대해 다음 순서로 실행합니다.

1. 질문으로 실제 검색을 수행합니다.
2. 검색 결과를 임시 LangChain LLM에 전달해 답변을 생성합니다.
3. 후보셋의 `expected_facts`를 참조 답변으로 저장합니다.
4. 후보셋의 `expected_source_chunk_ids`에 해당하는 chunk를 참조 컨텍스트로 저장합니다.

Ragas가 사용하는 필드는 다음과 같습니다.

- `user_input`: 사용자 질문
- `response`: RAG가 실제로 생성한 답변
- `retrieved_contexts`: 실제 검색 결과
- `reference`: 기대 사실을 합친 참조 답변
- `reference_contexts`: 정답 chunk로 구성한 참조 컨텍스트

특히 `retrieved_contexts`와 `reference_contexts`를 섞지 않는 것이 중요합니다. 그래야 Context Precision과 Context Recall이 검색 품질을 제대로 측정합니다.

In [4]:
records=[]
for s in samples:
    # 실제 RAG 검색 결과입니다. 정답 chunk를 대신 넣으면 안 됩니다.
    retrieved=retrieve(s["question"]); refs=[chunk_by_id[x] for x in s.get("expected_source_chunk_ids", []) if x in chunk_by_id]
    records.append({"case_id":s.get("case_id"), "user_input":s["question"], "response":answer_generator(s["question"], retrieved),
                    "retrieved_contexts":[context_text(c) for c in retrieved], "reference":" ".join(s.get("expected_facts", [])),
                    "reference_contexts":[context_text(c) for c in refs], "retrieved_chunk_ids":[c.get("chunk_id") for c in retrieved],
                    "expected_chunk_ids":s.get("expected_source_chunk_ids", [])})
run_df=pd.DataFrame(records); display(run_df[["case_id","retrieved_chunk_ids","expected_chunk_ids","response"]].head())

,case_id,retrieved_chunk_ids,expected_chunk_ids,response
0,CAND_0001,"[CHK_990840CF1BE502A6, CHK_8C6F024836C906F9, C...",[CHK_5509F5B4B0BA7701],"고추에서 탄저병 방제용으로 등록된 약제의 품목명, 사용 시기, 간격, 최대 사용 횟..."
1,CAND_0002,"[CHK_D2B418E04D6784E9, CHK_B77B2338D5424E62, C...",[CHK_6BB6B0B061444A54],INSUFFICIENT_EVIDENCE
2,CAND_0003,"[CHK_F982512051C97F18, CHK_797413559F100196, C...",[CHK_F982512051C97F18],"양파의 어린 유묘에서 생육이 정지되고 지상부가 시들며, 인경 내부에 검은색 가루가 ..."
3,CAND_0004,"[CHK_FD4C7E1491F6E9B2, CHK_F827B63A07FC46F6, C...",[CHK_209C451196282150],배추좀나방 방제를 위한 배추에 등록된 약제의 품목과 정식 전 사용 조건(희석배수와 ...
4,CAND_0005,"[CHK_F354FE0212ACE8F5, CHK_616917125E4DF199, C...",[CHK_0BA14A1A4BCD27C8],INSUFFICIENT_EVIDENCE\n\n컨텍스트에는 배추좀나방에 대해 등록된 ...


## 평가 실행

실제 검색 결과는 `retrieved_contexts`, 후보셋의 정답은 `reference`와 `reference_contexts`에만 넣습니다. 따라서 검색과 생성 성능이 분리됩니다.

## 5단계: 6개 지표 실행

Ragas는 평가용 LLM과 임베딩 모델을 사용해 각 질문을 채점합니다. 점수는 일반적으로 0~1 범위이며 높을수록 좋습니다.

- **Faithfulness / 답변 충실도**: 답변의 주장들이 검색된 컨텍스트에 의해 뒷받침되는가?
- **Answer Relevancy / 답변 관련성**: 답변이 질문에 직접 답하고 불필요하게 벗어나지 않는가?
- **Answer Correctness / 답변 정확성**: 답변이 `expected_facts`라는 참조 답변과 사실상 일치하는가?
- **Context Precision / 컨텍스트 정밀도**: 검색된 컨텍스트 중 관련 있는 chunk가 상위에 얼마나 많이 배치되었는가?
- **Context Recall / 컨텍스트 재현율**: 정답 컨텍스트에 필요한 근거를 검색 결과가 얼마나 회수했는가?
- **Context Relevance / 컨텍스트 관련성**: 검색된 컨텍스트가 질문과 얼마나 관련 있는가?

마지막으로 질문별 점수와 전체 평균을 표시합니다. 평균만 보지 말고 `case_id`별 점수와 검색된 chunk ID도 함께 확인해야 특정 유형의 질문에서 실패하는 원인을 찾을 수 있습니다.

In [5]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextPrecision, ContextRecall, ContextRelevance
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings
evaluator_llm=LangchainLLMWrapper(ChatOpenAI(model=MODEL_NAME, temperature=0))
evaluator_embeddings=LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))
ds=Dataset.from_pandas(run_df[["user_input","response","retrieved_contexts","reference","reference_contexts"]], preserve_index=False)
metrics=[Faithfulness(),AnswerRelevancy(),AnswerCorrectness(),ContextPrecision(),ContextRecall(),ContextRelevance()]
# 6개 지표를 동시에 계산합니다. 각 지표는 검색 결과와 참조값을 비교합니다.
result=evaluate(dataset=ds, metrics=metrics, llm=evaluator_llm, embeddings=evaluator_embeddings)
result_df=result.to_pandas(); result_df.insert(0,"case_id",run_df["case_id"].values)
display(result_df); display(result_df.select_dtypes("number").mean().to_frame("mean").T)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_31752\3517237582.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextPrecision, ContextRecall, ContextRelevance
C:\Users\Playdata\AppData\Local\Temp\ipykernel_31752\3517237582.py:3: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextPrecision, ContextRecall, ContextRelevance
C:\Users\Playdata\AppData\Local\Temp\ipykernel_31752\3517237582.py:3: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be rem

Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


,case_id,user_input,retrieved_contexts,reference_contexts,response,reference,faithfulness,answer_relevancy,answer_correctness,context_precision,context_recall,nv_context_relevance
0,CAND_0001,"고추에서 탄저병 방제용으로 등록된 약제의 품목명은 무엇이며, 발병 후 언제부터 어떤...",[[chunk_id=CHK_990840CF1BE502A6]\n발생조건: 병원균은 병...,[[chunk_id=CHK_5509F5B4B0BA7701]\n작물=고추; 병해충=탄...,"고추에서 탄저병 방제용으로 등록된 약제의 품목명, 사용 시기, 간격, 최대 사용 횟...",고추의 탄저병 방제용 등록 품목은 베노밀 수화제이다 발병 초부터 10일 간격으로 사...,1.000000,0.376102,0.559081,0.200000,1.000,1.00
1,CAND_0002,"배추를 고온기에 엇갈이 재배할 때 선택해야 하는 품종군은 무엇이며, 어떤 특성이 강...",[[chunk_id=CHK_D2B418E04D6784E9]\n발생조건: CMV는 토...,"[[chunk_id=CHK_6BB6B0B061444A54]\n{""characteri...",INSUFFICIENT_EVIDENCE,배추에는 내서·내습·내병성 품종군이 있다 이 품종군은 고온기 엇갈이 재배에서 내서성...,0.000000,0.000000,0.017663,0.000000,0.000,0.00
2,CAND_0003,"양파의 어린 유묘에서 생육이 정지되고 지상부가 시들며, 인경 내부에 검은색 가루가 ...",[[chunk_id=CHK_F982512051C97F18]\n발생조건: 병원균은 토...,[[chunk_id=CHK_F982512051C97F18]\n발생조건: 병원균은 토...,"양파의 어린 유묘에서 생육이 정지되고 지상부가 시들며, 인경 내부에 검은색 가루가 ...",양파는 깜부기병의 영향을 받는다,0.000000,0.969922,0.120066,1.000000,1.000,1.00
3,CAND_0004,"배추좀나방을 방제하려고 하는데, 배추에 등록된 약제의 품목과 정식 전 사용 조건(희...",[[chunk_id=CHK_FD4C7E1491F6E9B2]\n작물=배추; 병해충=일...,[[chunk_id=CHK_209C451196282150]\n작물=배추; 병해충=배...,배추좀나방 방제를 위한 배추에 등록된 약제의 품목과 정식 전 사용 조건(희석배수와 ...,배추좀나방 방제용으로 배추에 등록된 품목은 아바멕틴·디노테퓨란 액제이다. 사용적기와...,0.800000,0.000000,0.505730,0.000000,0.000,0.00
4,CAND_0005,배추에서 배추좀나방이 발생하기 시작했을 때 사용할 수 있도록 등록된 농약 품목은 무...,[[chunk_id=CHK_F354FE0212ACE8F5]\n【부록】 국내 유기농업...,[[chunk_id=CHK_0BA14A1A4BCD27C8]\n작물=배추; 병해충=배...,INSUFFICIENT_EVIDENCE\n\n컨텍스트에는 배추좀나방에 대해 등록된 ...,배추좀나방 방제용으로 배추에 등록된 품목은 루페뉴론 유제(상표명 파이터)이다 사용적...,1.000000,0.000000,0.099818,0.000000,0.000,0.25
5,CAND_0006,고추 과실에 구멍을 뚫고 속을 파먹어 피해를 주는 해충은 무엇인가요?,[[chunk_id=CHK_3D99774ADB173915]\n피해: 담배나방의 유충...,[[chunk_id=CHK_3D99774ADB173915]\n피해: 담배나방의 유충...,고추 과실에 구멍을 뚫고 속을 파먹어 피해를 주는 해충은 담배나방(Helicover...,고추는 담배나방의 피해를 받는다,1.000000,0.997067,0.287385,1.000000,1.000,1.00
6,CAND_0007,"고온기에 배추를 엇갈이 재배하려고 하는데, 품종을 고를 때 어떤 재배 적응성을 중점...","[[chunk_id=CHK_6BB6B0B061444A54]\n{""characteri...","[[chunk_id=CHK_6BB6B0B061444A54]\n{""characteri...","고온기에 배추를 엇갈이 재배할 때 품종을 고를 때는 내서성(더위 견디는 성질), 내...",고온기 엇갈이 재배에서는 내서성이 강한 품종을 선택해야 한다 고온기 엇갈이 재배에서...,0.571429,0.798298,0.640320,0.700000,1.000,1.00
7,CAND_0008,시설재배 배추에 비료를 줄 때 밑거름과 웃거름의 질소·인산·칼리 표준 성분량은 각각...,[[chunk_id=CHK_E8FC629E9850B792]\n배추(시설재배) 비료 ...,[[chunk_id=CHK_E8FC629E9850B792]\n배추(시설재배) 비료 ...,시설재배 배추에 비료를 줄 때 밑거름과 웃거름의 질소·인산·칼리 표준 성분량(kg/...,"밑거름 표준 성분량은 질소 6.2kg/10a, 인산 3.0kg/10a, 칼리 4.0...",1.000000,0.850610,0.934619,1.000000,1.000,1.00
8,CAND_0009,2023년 12월 19일 경상남도 진주시의 농가·육묘장에서 파종한 상추에는 어떤 품...,[[chunk_id=CHK_DA5EE70F4F882554]\n경상남도 진주시의 익명...,[[chunk_id=CHK_DA5EE70F4F882554]\n경상남도 진주시의 익명...,2023년 12월 19일 경상남도 진주시의 농가·육묘장에서 파종한 상추에는 '공심채...,파종에 사용된 상추 품종은 공심채이다.,1.000000,0.933655,0.637290,1.000000,1.000,1.00
9,CAND_0010,"마늘의 뿌리응애 방제에 등록된 농약 품목은 무엇이며, 사용 전에 어떤 등록 상태를 ...",[[chunk_id=CHK_616917125E4DF199]\n(3) 방제법 - 우량...,[[chunk_id=CHK_F90100052F0DAA74]\n작물=마늘; 병해충=뿌...,"마늘의 뿌리응애 방제에 등록된 농약 품목은 ""페니트로티온 유제(상표명: 일다해, 등...",등록 품목은 비펜트린 액상수화제이며 상표는 히든탄이다 검증상태가 REVIEW_REQ...,0.800000,0.863006,0.108113,0.000000,0.000,1.00


,faithfulness,answer_relevancy,answer_correctness,context_precision,context_recall,nv_context_relevance
mean,0.744762,0.488194,0.403318,0.414444,0.538333,0.733333


## 6단계: 결과 저장

질문별 점수를 JSONL로 저장합니다. 이후 CSV 변환, 유형별 평균(`question_type`), 난이도별 평균, 실패 사례 분석 등에 재사용할 수 있습니다.

주의할 점은 평가 점수가 LLM의 판단을 포함한다는 것입니다. 따라서 모델 버전, 프롬프트, `TOP_K`, 평가 대상 모델이 바뀌면 점수도 달라질 수 있습니다. 비교 실험에서는 이 설정들을 동일하게 유지해야 합니다.

In [6]:
OUTPUT_PATH=ROOT / "data/evaluation/rag_eval_results_candidates유건.jsonl"
result_df.to_json(OUTPUT_PATH, orient="records", force_ascii=False, lines=True)
print(f"saved: {OUTPUT_PATH}")

saved: c:\Users\Playdata\Desktop\mle-01-p3-team2\data\evaluation\rag_eval_results_candidates유건.jsonl
